# REPT — long-horizon simulations (2000 cycles)

Each of these 6 cells has a calibrated D_SEI (from the workbook b1→b2 fade,
see `04_rept_cohort_results.ipynb`). Here we extend the PyBaMM simulation
to **2000 cycles** to see (a) whether the short-window calibration extrapolates
honestly and (b) what cycle the model projects each cell to hit end-of-life
(SoH = 80 %).

**Key thing to remember**: the longterm CSVs only cover the first ~150 cycles.
Beyond that the comparison is sim-only — there's no measured ground truth at
cycle 2000.

Cells covered: **REPT_78, REPT_87, REPT_7, REPT_43, REPT_74, REPT_3**

Run `Voltaris/scripts/rept_long_sim.py --n-cycles 2000` first to regenerate
artifacts.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO_ROOT = Path("/home/hj/Desktop/PINNs").resolve()
OUT_DIR = REPO_ROOT / "Voltaris/outputs/tuned_params"
LONGTERM_DIR = REPO_ROOT / "Data/Longterm"
CELLS = [78, 87, 7, 43, 74, 3]
SKIP_FIRST_N = 4
N_CYCLES_LONG = 2000
N_CYCLES_MEASURED_AVAILABLE = 150   # CSV horizon — stop measured plots here
EOL_THRESHOLD = 80.0                 # SoH (%) for EOL projection
print(f"Reading from {OUT_DIR}")

## 1. Summary table — sim trajectory + projected EOL

Columns:
- `sim @ cy 150` / `cy 500` / `cy 1000` / `cy 2000`: simulated SoH at landmark cycles
- `meas @ cy 150`: measured SoH at the end of the longterm CSV
- `EOL cycle`: cycle at which the sim crosses SoH = 80 % (or "> 2000" if it doesn't)

In [ ]:
def _eol_cycle(sim_df: pd.DataFrame, threshold: float = EOL_THRESHOLD) -> float:
    """Cycle at which the sim's SoH first crosses `threshold`. Returns NaN if
    the sim never reaches that low — but caller should treat it as '> N_max'."""
    soh_pct = sim_df["SOH"].values * 100.0
    cyc     = sim_df["cycle_n"].values.astype(float)
    below = np.where(soh_pct < threshold)[0]
    if below.size == 0:
        return float("nan")
    # Linear-interp between the cycle just before and at the crossing
    i = below[0]
    if i == 0:
        return float(cyc[0])
    x0, x1 = cyc[i - 1], cyc[i]
    y0, y1 = soh_pct[i - 1], soh_pct[i]
    return float(x0 + (threshold - y0) * (x1 - x0) / (y1 - y0))


rows = []
for cid in CELLS:
    tag = f"REPT_{cid}"
    aging = json.loads((OUT_DIR / f"{tag}_aging_calibrated.json").read_text())
    sim_df = pd.read_parquet(OUT_DIR / f"{tag}_long_sim_{N_CYCLES_LONG}cy.parquet")

    csv = LONGTERM_DIR / f"REPT_Longterm_cell_{int(cid):04d}.csv"
    raw = pd.read_csv(csv, usecols=["cycle_no", "step_name", "capacity_ah"])
    dchg = raw[raw.step_name.astype(str).str.contains("DChg")]
    per_cyc = (dchg.groupby("cycle_no").capacity_ah
                    .agg(lambda s: float(s.abs().max())).reset_index())
    nominal = 150.0
    per_cyc["soh_pct"] = per_cyc.capacity_ah / nominal * 100.0

    # SoH at landmark cycles (interp into the sim curve)
    sim_soh = sim_df["SOH"].values * 100.0
    sim_cy  = sim_df["cycle_n"].values.astype(float)
    def at(n): return float(np.interp(n, sim_cy, sim_soh))

    eol = _eol_cycle(sim_df, EOL_THRESHOLD)

    rows.append({
        "cell": tag,
        "D_SEI (m²/s)": aging["calibrated_value"],
        "workbook tgt (pp/100cy)": aging["target_slope_pp_per_100cy"],
        "sim @ cy 150":  at(150),
        "sim @ cy 500":  at(500),
        "sim @ cy 1000": at(1000),
        "sim @ cy 2000": at(2000),
        "meas @ cy 150": float(per_cyc.soh_pct.iloc[-1]),
        "EOL cycle (sim)": eol if not np.isnan(eol) else float("inf"),
    })
df = pd.DataFrame(rows)
display(df.style.format({
    "D_SEI (m²/s)": "{:.3e}",
    "workbook tgt (pp/100cy)": "{:+.4f}",
    "sim @ cy 150":  "{:.2f}",
    "sim @ cy 500":  "{:.2f}",
    "sim @ cy 1000": "{:.2f}",
    "sim @ cy 2000": "{:.2f}",
    "meas @ cy 150": "{:.2f}",
    "EOL cycle (sim)": lambda v: "> 2000" if v == float("inf") else f"{v:.0f}",
}))

## 2. Combined 2×3 grid — full 2000-cycle sim with measured CSV overlay

Anchored at cycle 5 (first 4 cycles skipped). The red trace stops at cycle ~150
(end of longterm CSV); the blue trace continues to cycle 2000. A dotted black
line marks the EOL threshold (SoH = 80 %)."

In [ ]:
def _drop_outliers(series: pd.Series, k: float = 3.0, window: int = 5) -> pd.Series:
    if len(series) < window:
        return pd.Series([True] * len(series), index=series.index)
    med = series.rolling(window, center=True, min_periods=1).median()
    mad = (series - med).abs().rolling(window, center=True, min_periods=1).median()
    threshold = k * 1.4826 * mad.clip(lower=1e-9)
    return (series - med).abs() <= threshold


fig, axs = plt.subplots(2, 3, figsize=(16, 8))
axs = axs.flatten()
for i, cid in enumerate(CELLS):
    ax = axs[i]
    tag = f"REPT_{cid}"
    aging = json.loads((OUT_DIR / f"{tag}_aging_calibrated.json").read_text())
    sim_df = pd.read_parquet(OUT_DIR / f"{tag}_long_sim_{N_CYCLES_LONG}cy.parquet")
    csv = LONGTERM_DIR / f"REPT_Longterm_cell_{int(cid):04d}.csv"
    raw = pd.read_csv(csv, usecols=["cycle_no","step_name","capacity_ah"])
    dchg = raw[raw.step_name.astype(str).str.contains("DChg")]
    per_cyc = (dchg.groupby("cycle_no").capacity_ah
                    .agg(lambda s: float(s.abs().max())).reset_index())
    per_cyc["soh_pct"] = per_cyc.capacity_ah / 150.0 * 100.0

    m = per_cyc[per_cyc.cycle_no >= SKIP_FIRST_N + 1].copy()
    keep = _drop_outliers(m["soh_pct"], k=3.0, window=5)
    clean   = m[keep]
    dropped = m[~keep]

    sim_soh = sim_df["SOH"].values * 100.0
    sim_cy  = sim_df["cycle_n"].values.astype(float)
    first_cy = int(clean.cycle_no.iloc[0])
    meas_start = float(clean["soh_pct"].iloc[0])
    sim_anchored = sim_soh + (meas_start - sim_soh[0])
    sim_cy_anchored = sim_cy + (first_cy - sim_cy[0])

    ax.plot(clean.cycle_no, clean["soh_pct"], "o-", lw=0.7, ms=2,
            color="#d62728",
            label=f"measured (CSV, ends cy {int(clean.cycle_no.max())})")
    if not dropped.empty:
        ax.scatter(dropped.cycle_no, dropped["soh_pct"], s=30,
                    marker="x", color="grey", alpha=0.5, zorder=2)
    ax.plot(sim_cy_anchored, sim_anchored, "-", lw=1.0,
            color="#1f77b4",
            label=f"sim (anchored, {N_CYCLES_LONG} cy)")
    ax.axhline(EOL_THRESHOLD, ls=":", color="black", alpha=0.5,
                label=f"EOL = {int(EOL_THRESHOLD)} %")

    rel = aging.get("relative_error_pct", float("nan"))
    ax.set_title(f"{tag} — D_SEI={aging['calibrated_value']:.1e}, err={rel:.1f}%",
                  fontsize=10)
    ax.set_xlabel("Cycle"); ax.set_ylabel("SoH (%)")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=8)

fig.suptitle(f"REPT — {N_CYCLES_LONG}-cycle simulations vs longterm CSV "
              f"(measured ends ~cy {N_CYCLES_MEASURED_AVAILABLE})",
              fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

## 3. Three slopes per cell — the calibration-vs-measured-window story

For each cell:
- **workbook target** = the slope the calibration was tuned to (from b1→b2 SoH, averaged over ~600 cy)
- **sim slope (early 150 cy)** = what PyBaMM produced at the calibrated D_SEI over the same 150-cy window the CSV covers
- **CSV slope** = the actual measured early-cycle slope

The sim slope is computed over the **first 150 cycles** of the 2000-cy run so it's directly comparable to the CSV. The full 2000-cy sim slope is shallower (SEI growth is sublinear)."

In [ ]:
# Re-compute slopes — sim window mirrors the CSV horizon for a fair comparison.
slope_rows = []
for cid in CELLS:
    tag = f"REPT_{cid}"
    aging = json.loads((OUT_DIR / f"{tag}_aging_calibrated.json").read_text())
    sim_df = pd.read_parquet(OUT_DIR / f"{tag}_long_sim_{N_CYCLES_LONG}cy.parquet")
    csv = LONGTERM_DIR / f"REPT_Longterm_cell_{int(cid):04d}.csv"
    raw = pd.read_csv(csv, usecols=["cycle_no","step_name","capacity_ah"])
    dchg = raw[raw.step_name.astype(str).str.contains("DChg")]
    per_cyc = (dchg.groupby("cycle_no").capacity_ah
                    .agg(lambda s: float(s.abs().max())).reset_index())
    per_cyc["soh_pct"] = per_cyc.capacity_ah / 150.0 * 100.0
    meas_window = per_cyc[per_cyc.cycle_no >= SKIP_FIRST_N + 1]

    sim_early = sim_df[sim_df.cycle_n <= N_CYCLES_MEASURED_AVAILABLE]

    slope_rows.append({
        "cell": tag,
        "workbook tgt": aging["target_slope_pp_per_100cy"],
        "sim early (≤150 cy)": float(np.polyfit(sim_early.cycle_n[1:],
                                                  sim_early.SOH[1:] * 100.0, 1)[0] * 100),
        "sim full (2000 cy)":  float(np.polyfit(sim_df.cycle_n[1:],
                                                  sim_df.SOH[1:] * 100.0, 1)[0] * 100),
        "CSV (≤150 cy)":       float(np.polyfit(meas_window.cycle_no,
                                                  meas_window.soh_pct, 1)[0] * 100),
    })
sdf = pd.DataFrame(slope_rows)
display(sdf.style.format("{:+.4f}", subset=sdf.columns.drop("cell")))

fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(sdf))
w = 0.2
ax.bar(x - 1.5*w, sdf["workbook tgt"],         w, label="workbook target (b1→b2)", color="#1f77b4")
ax.bar(x - 0.5*w, sdf["sim early (≤150 cy)"],  w, label="sim ≤150 cy",             color="#9ecae1")
ax.bar(x + 0.5*w, sdf["sim full (2000 cy)"],   w, label="sim full 2000 cy",        color="#2ca02c")
ax.bar(x + 1.5*w, sdf["CSV (≤150 cy)"],        w, label="longterm CSV ≤150 cy",    color="#d62728")
ax.set_xticks(x); ax.set_xticklabels(sdf["cell"], rotation=20)
ax.set_ylabel("slope (pp / 100 cy)")
ax.axhline(0, ls="-", color="black", lw=0.6)
ax.set_title("REPT — fade slopes by source / window")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4. Projected EOL cycle (sim crosses SoH = 80 %)

Where each cell's simulated trajectory crosses the EOL threshold. Cells whose
projected EOL is beyond the 2000-cycle horizon are reported as `> 2000` —
the calibrated D_SEI predicts they'd survive substantially longer."

In [ ]:
eol_values = df["EOL cycle (sim)"].copy()
# Replace inf with N_CYCLES_LONG so they still draw something visible
eol_plot = eol_values.replace(float("inf"), N_CYCLES_LONG)
finite_mask = ~np.isinf(eol_values)

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ["#d62728" if v == float("inf") else "#1f77b4" for v in eol_values]
ax.barh(df["cell"], eol_plot, color=colors)
ax.axvline(N_CYCLES_LONG, ls="--", color="grey", alpha=0.6,
            label=f"sim horizon ({N_CYCLES_LONG} cy)")
for j, (cell, v) in enumerate(zip(df["cell"], eol_values)):
    label = "> 2000" if v == float("inf") else f"{v:.0f}"
    ax.text(min(v, N_CYCLES_LONG) + 30, j, label, va="center", fontsize=9)
ax.set_xlabel("projected EOL cycle (sim crosses SoH = 80 %)")
ax.set_title("REPT — projected EOL from calibrated D_SEI (sim horizon = 2000 cy)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("Cells whose sim never reaches EOL within 2000 cy:",
       df.loc[np.isinf(eol_values), "cell"].tolist() or "—")

## 5. Diagnosis

**Two things to take from these 2000-cycle simulations:**

### A. The early-window (≤150 cy) calibration vs measured story
- The 150-cy sim slope is dictated by the workbook target (calibration was tuned against it). For most cells the workbook target is a few-tenths pp/100cy.
- The CSV slope is ~2–3× steeper for most cells; cell **REPT_7** is the standout where the cell *really* aged faster than the workbook average suggested.
- This confirms the cohort-level finding from notebook 04: the workbook averages over ~600 cycles, while the measured CSV is the steeper early-cycle slice. They diverge cell-by-cell.

### B. The 2000-cycle long-horizon story
- D_SEI is *constant* in the SEI-solvent-diffusion-limited model, but the SoH(cycle) curve isn't linear — fade rate slows over time because the SEI layer thickens and limits further reaction. The "sim full (2000 cy)" slope is much shallower than the "sim early (≤150 cy)" slope for that reason.
- The EOL projection is therefore optimistic (i.e. cell survives longer than a naive linear extrapolation of the 150-cy slope would suggest).
- Because the workbook target is itself a slow-aging signal, the projected EOL for several of these cells is `> 2000` — the calibration says they'd outlast the simulated horizon by a wide margin. That's likely too generous; the real long-horizon fade probably includes mechanisms (LAM, plating) the SEI-only calibration ignores.

### What this means for downstream PINN training
- The calibrated D_SEI values are *internally consistent* with the workbook fade rate and reproduce the correct order-of-magnitude SoH(cycle) shape.
- For absolute EOL prediction beyond ~500 cy these values should be cross-checked against any aged cell that *actually* hit SoH=80 % in the lab.
- A natural next step is a second calibration mode that fits to the longterm CSV slope directly (target = CSV early-cycle slope), and compare the resulting D_SEI distribution. That's the divergence we'd want a PINN to absorb as cell-level variation."